In [ ]:
%pip install requests pyarrow pandas numpy numba plotly huggingface_hub

In [ ]:
import os
import sys

# 1. Define your GitHub repository details
REPO_URL = "https://github.com/payamdav/pycrypto.git"
REPO_NAME = "pycrypto"
BRANCH_NAME = "claude/charming-hawking-y5j5y6"

# 2. Clone the specific branch if it hasn't been cloned yet
if not os.path.exists(REPO_NAME):
    !git clone -b {BRANCH_NAME} {REPO_URL}

# 3. Add the cloned repository root to the Python path
REPO_PATH = os.path.abspath(REPO_NAME)
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

In [ ]:
import os
import sys
import io
import re
import requests
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import plotly.graph_objects as go
from packages.kalman_filter.kalman_fast import kalman_filter_batch

In [ ]:
asset      = "btcusdt"       # lowercase asset name
start_date = "2025-01-01"   # download range start
end_date   = "2025-03-31"   # download range end

In [ ]:
from datetime import datetime, timezone
from huggingface_hub import HfApi

REPO = "https://huggingface.co/datasets/payamdavaee/candles/resolve/main"

def load_range(asset, start, end, columns=None):
    api = HfApi()
    files = api.list_repo_files("payamdavaee/candles", repo_type="dataset")
    pattern = re.compile(rf"^{asset}/{asset}-1m-(\d{{4}})-(\d{{2}})\.parquet$")
    start_ts = int(datetime.fromisoformat(start).replace(tzinfo=timezone.utc).timestamp() * 1000)
    end_ts   = int(datetime.fromisoformat(end).replace(tzinfo=timezone.utc).timestamp() * 1000) + 86_400_000
    frames = []
    for f in sorted(files):
        m = pattern.match(f)
        if not m:
            continue
        url = f"{REPO}/{f}"
        resp = requests.get(url, timeout=60)
        tbl = pq.read_table(io.BytesIO(resp.content), columns=columns or None)
        df_ = tbl.to_pandas()
        ts = df_["ts"].astype("int64")
        df_ = df_[(ts >= start_ts) & (ts < end_ts)]
        if not df_.empty:
            frames.append(df_)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames).sort_values("ts").reset_index(drop=True)

df = load_range(asset, start_date, end_date, columns=["ts", "vwap"])
df["ts"] = pd.to_datetime(df["ts"], unit="ms", utc=True)

print(f"Shape: {df.shape}")
print(df.head())

In [ ]:
print("VWAP statistics:")
print(df["vwap"].describe())
print()

date_min = df["ts"].min()
date_max = df["ts"].max()
total_days = (date_max - date_min).days + 1
expected_minutes = total_days * 1440
actual_count = len(df)
missing_minutes = expected_minutes - actual_count

print(f"Date range : {date_min} -> {date_max}")
print(f"Total days : {total_days}")
print(f"Expected rows (1-min candles): {expected_minutes}")
print(f"Actual rows                  : {actual_count}")
print(f"Missing minutes              : {missing_minutes}")

In [ ]:
plot_start_date      = "2025-01-01"   # must be within downloaded range
plot_end_date        = "2025-01-31"
initial_estimate     = 0.0            # x̂_0  (0.0 => auto-set to first vwap value)
initial_error_cov    = 1.0            # P_0
process_variance     = 1e-4           # Q
measurement_variance = 1e-2           # R
k                    = 2.0            # band multiplier; set to 0 to disable bands

In [ ]:
import pandas as pd

# Filter to plot range
plot_start = pd.Timestamp(plot_start_date, tz="UTC")
plot_end   = pd.Timestamp(plot_end_date,   tz="UTC") + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
df_plot = df[(df["ts"] >= plot_start) & (df["ts"] <= plot_end)].copy()
df_plot = df_plot.reset_index(drop=True)

# Extract measurements
measurements = df_plot["vwap"].to_numpy(dtype=np.float64)

# Auto-set initial estimate if zero
x0 = measurements[0] if initial_estimate == 0.0 else float(initial_estimate)

# Run Kalman filter
estimates, error_covariances = kalman_filter_batch(
    measurements,
    x0,
    float(initial_error_cov),
    float(process_variance),
    float(measurement_variance)
)

timestamps = df_plot["ts"]

# Build figure
fig = go.Figure()

# Raw VWAP
fig.add_trace(go.Scatter(
    x=timestamps,
    y=measurements,
    mode="lines",
    name="VWAP",
    line=dict(color="rgba(100,149,237,0.5)", width=1)
))

# Kalman estimate
fig.add_trace(go.Scatter(
    x=timestamps,
    y=estimates,
    mode="lines",
    name="Kalman Filter",
    line=dict(color="#00E5FF", width=2)
))

if k > 0:
    upper_band = estimates + k * error_covariances
    lower_band = estimates - k * error_covariances

    # Lower band (added first so fill='tonexty' fills up to upper)
    fig.add_trace(go.Scatter(
        x=timestamps,
        y=lower_band,
        mode="lines",
        name=f"Lower Band (k={k})",
        line=dict(color="rgba(100,255,100,0.7)", width=1, dash="dash")
    ))

    # Upper band with fill back to lower band
    fig.add_trace(go.Scatter(
        x=timestamps,
        y=upper_band,
        mode="lines",
        name=f"Upper Band (k={k})",
        line=dict(color="rgba(255,100,100,0.7)", width=1, dash="dash"),
        fill="tonexty",
        fillcolor="rgba(150,150,150,0.1)"
    ))

fig.update_layout(
    template="plotly_dark",
    title=f"Kalman Filter — {asset.upper()} VWAP ({plot_start_date} \u2192 {plot_end_date})",
    height=600,
    xaxis_title="Time (UTC)",
    yaxis_title="Price (USDT)",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()